# Data Sources:

## CU Boulder Online MSCS Spreadsheet

Compiled and maintained by Justin Ong (alias "Razuki" in the student Slack channel and unofficial Discord server)

https://docs.google.com/spreadsheets/d/1lplPW_5DI-wgB_q6qgxmr9WTP12-JVV_yDXsKxLFMiM/edit?gid=1698402210#gid=1698402210

Accessed: 12/9/2024

Extracted spreadsheet tabs, saved as individual .csv files (supplemental headers stripped):
* MSCS Review Responses.csv *(hidden in original)*
* Outside Review Form Responses.csv *(hidden in original)*
* MSCS Info.csv
* MSDS Info.csv
* MSEE Info.csv
* MEEM Info.csv


## Supplemental Data:

CU Boulder Online MSCS Student Handbooks, 2023-2024 and 2024-2025

* 2023-2024: https://www.colorado.edu/cs/sites/default/files/attached-files/ms-cs_on_coursera_handbook_-_2023.05.18.pdf
* 2024-2025: https://www.colorado.edu/cs/media/155

These were used to manually categorize certain features further in the dataset into a .csv file:
* mscs_requirements.csv



In [1]:
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np



In [2]:
data_dir = Path(r"C:\Users\james\OneDrive\study\ML\ml1\ml1_final_project_data")

# Data Cleaning


In [3]:
mscs_reviews = pd.read_csv(data_dir / 'MSCS Review Responses.csv', parse_dates=['Timestamp'])
outside_reviews = pd.read_csv(data_dir / 'Outside Review Form Responses.csv', parse_dates=['Timestamp'])
mscs_info = pd.read_csv(data_dir / 'MSCS Info.csv')
msds_info = pd.read_csv(data_dir / 'MSDS Info.csv')
msee_info = pd.read_csv(data_dir / 'MSEE Info.csv')
meem_info = pd.read_csv(data_dir / 'MEEM Info.csv')
mscs_requirements = pd.read_csv(data_dir / 'mscs_requirements.csv')

In [4]:
all_dfs = [mscs_reviews, outside_reviews, mscs_info, msds_info, msee_info, meem_info]
null_fixes = {
    'Null': None,
}
for i, df in enumerate(all_dfs):
    # Clean up 'Null' (str) to None.
    df = df.replace(to_replace=null_fixes)
    all_dfs[i] = df
# Unpack back to original variables.
mscs_reviews, outside_reviews, mscs_info, msds_info, msee_info, meem_info = all_dfs


Fix values in `'Finals Weightage'` that aren't clean percentages, and convert them all to integers.

In [5]:
# Clean final weightage percentages, and convert to int in a new column.
info_dfs = [mscs_info, msds_info, msee_info, meem_info]
weightage_fixes = {
    '22% For Capstone + 22% for Final Exam': '44%',
    '10% / 15%': '25%',
    '?': np.NaN,
}
for i, df in enumerate(info_dfs):
    df['Finals Weightage'] = df['Finals Weightage'].replace(to_replace=weightage_fixes)
    df['final_weightage_pct'] = df['Finals Weightage'].replace(to_replace={np.NaN: '0%', None: '-1%'})
    df['final_weightage_pct'] = df['final_weightage_pct'].apply(lambda s: int(s.strip('%')))
    df['final_weightage_pct'] = df['final_weightage_pct'].replace(to_replace={0: np.NaN, -1: None})
    info_dfs[i] = df
# Unpack back to original variables.
mscs_info, msds_info, msee_info, meem_info = info_dfs


Using the student handbooks, classes within the MSCS program were further categorized as follows:

* `'2023_category'`: `'Pathway'`, `'Breadth'`, or `'Elective'` (for 2023-2024 students)
* `'2024_category'`: `'Pathway'`, `'Breadth'`, `'Elective'`, or `'Disallowed'` (for 2024-2025 students)
* `'2023_required'`: Whether the class is required to complete the degree for 2023-2024 students. (`True` or `False`)
* `'2024_required'`: Whether the class is required to complete the degree for 2024-2025 students. (`True` or `False`)

This data was saved to `mscs_requirements.csv`.

In [6]:
# Join mscs_info with mscs_requirements.
mscs_requirements = mscs_requirements.drop(columns=['Course Title'])
mscs_info = mscs_info.merge(mscs_requirements, on='Code', how='left')

All of the outside electives are optional.

In [7]:
# Add features to outside info df's; clean up; and concat into one.
outside_info_dfs = [msds_info, msee_info, meem_info]
standardized_col_names = {
    'Finals Details.1': 'Assignment Details',
    'Course Title': 'Course Name',
}
for i, df in enumerate(outside_info_dfs):
    # All outside classes are 'Outside Elective' and not required.
    df['2023_category'] = 'Outside Elective'
    df['2024_category'] = 'Outside Elective'
    df['2023_required'] = False
    df['2024_required'] = False
    df = df.rename(standardized_col_names, axis=1)
    outside_info_dfs[i] = df
# Unpack back to original variables.
msds_info, msee_info, meem_info = outside_info_dfs
outside_info = pd.concat(outside_info_dfs, join='outer')

The data includes information regarding what kind of final exam or assignment. Here, we'll parse it out into binary categories.

In [8]:
# Categorize final types.
assignment_type = {
    'Assignment',
    'Assignment/Exam',
}
project_type = {
    'Project',
    'Capstone project + Final Exam',
}
exam_type = {
    'Exam (Non-\nProctored)',
    'Exam (Non-proctored)',
    'Exam (Proctor)', 
    'Exam\n(Non-proctor)',
    'Capstone project + Final Exam',
    'Exam',
    'Assignment/Exam',
}
proctor_type = {
    'Exam (Proctor)',
}

info_dfs = [mscs_info, outside_info]

for i, df in enumerate(info_dfs):
    df['final_exam'] = df['Finals'].isin(exam_type)
    df['final_exam_proctored'] = df['Finals'].isin(proctor_type)
    df['final_assignment'] = df['Finals'].isin(assignment_type)
    df['final_project'] = df['Finals'].isin(project_type)
    info_dfs[i] = df

mscs_info, outside_info = info_dfs


In [9]:
outside_info

,Code,Course Name,Offered By,Hours,Language,Finals,Finals Weightage,Finals Details,Assignment Details,final_weightage_pct,2023_category,2024_category,2023_required,2024_required,final_exam,final_exam_proctored,final_assignment,final_project
0,DTSA 5001,Probability Theory: Applications for Data Science,Pathway - Statistical Inference,48.0,R,Exam (Proctor),22%,Two-hour exam; unlimited; 15 open-ended questions,NaN,22,Outside Elective,Outside Elective,False,False,True,True,False,False
1,DTSA 5002,Statistical Inference for Estimation in Data S...,Pathway - Statistical Inference,26.0,R,Exam (Proctor),20%,Two-hour exam; 1 attempt; 15 MCQ/open-ended qu...,NaN,20,Outside Elective,Outside Elective,False,False,True,True,False,False
2,DTSA 5003,Statistical Inference & Hypothesis Testing in ...,Pathway - Statistical Inference,34.0,R,Exam (Proctor),21%,Two-hour exam; 1 attempt; 15 MCQ/open-ended qu...,NaN,21,Outside Elective,Outside Elective,False,False,True,True,False,False
3,DTSA 5501,"Algorithms for Searching, Sorting, and Indexing",Pathway - Core CS - DSA,35.0,NaN,NaN,NaN,NaN,NaN,None,Outside Elective,Outside Elective,False,False,False,False,False,False
4,DTSA 5502,Trees and Graphs: Basics\n,Pathway - Core CS - DSA,34.0,NaN,NaN,NaN,NaN,NaN,None,Outside Elective,Outside Elective,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40,EMEA 5035,Applying Systems Engineering to the Design Pro...,MEEM,21.0,None,Exam,25%,NaN,NaN,25,Outside Elective,Outside Elective,False,False,True,False,False,False
41,EMEA 5036,Systems Engineering and Program Management,MEEM,15.0,None,Project,28%,NaN,NaN,28,Outside Elective,Outside Elective,False,False,False,False,False,True
42,EMEA 5216,Sustainability and the Circular Economy,MEEM,23.0,NaN,NaN,NaN,NaN,NaN,None,Outside Elective,Outside Elective,False,False,False,False,False,False
43,EMEA 5231,Resilience and Leadership: Concepts Definition...,MEEM,26.0,NaN,NaN,NaN,NaN,NaN,None,Outside Elective,Outside Elective,False,False,False,False,False,False


`Outside Review Form Responses.csv` has `'Course Name'` data in three identically named columns.

In [10]:
# Combine columns into 'Course Name' in the Outside Reviews data.
outside_reviews.loc[outside_reviews['Course Name'].isna(), 'Course Name'] = outside_reviews['Course Name.1']
outside_reviews.loc[outside_reviews['Course Name'].isna(), 'Course Name'] = outside_reviews['Course Name.2']
outside_reviews = outside_reviews.drop(columns=['Course Name.1', 'Course Name.2'])


In [11]:
# Simplify relevant feature names.
features_to_rename = {
    'Overall Experience: How would you rate your overall experience in the course?': 'overall_experience',
    'Course Content Difficulty: How would you rate the difficulty of the course content?': 'content_difficulty',
    'Instructor Effectiveness: How effective was the instructor in teaching the course?': 'instructor_effectiveness',
    'Time Commitment: Approximately how many hours did you spend on this course?': 'time_commitment_hours',
}
mscs_reviews = mscs_reviews.rename(features_to_rename, axis=1)
outside_reviews = outside_reviews.rename(features_to_rename, axis=1)

In [12]:
# Extract course 'Code' from the course name.
outside_reviews['Code'] = outside_reviews['Course Name'].apply(lambda s: s[:9])
mscs_reviews['Code'] = mscs_reviews['Course Name'].apply(lambda s: s[:9])

### Now the `info` data can be merged into the review data.

In [14]:
mscs_reviews = mscs_reviews.merge(mscs_info, on='Code', how='left', suffixes=(None, '_frominfodata'))
outside_reviews = outside_reviews.merge(outside_info, on='Code', how='left', suffixes=(None, '_frominfodata'))

The curriculum requirements for the MSCS program changed for the 2024-2025 academic year, which began 8/26/2024.
The reviews span the 2023-2024 and 2024-2025 academic years. I want to add a feature to __approximately__ capture
whether the class was required when it was taken by the reviewer.

A cutoff date of 9/10/2024 was used--i.e., it was assumed that any reviews left before that date were for classes
taken during the 2023-2024 academic year (allowing for a couple weeks after the term ended to get reviews in); and
any subsequent reviews were for classes taken during the 2024-2025 academic year.

Note that students who left reviews after that date might still be subject to the 2023-2024 curriculum requirements,
so there remains some uncertainty in the data.

In [15]:
# The 2024 Summer 2 term ended 8/23/2024; and the new 
# 2024-2025 curriculum began Fall 1 term.
# Set the cutoff date to be a couple weeks later, with a guess
# that it might have taken students a couple weeks to submit
# their reviews.
cutoff_2024 = datetime(2024, 9, 10)

In [16]:
# Determine whether a course was required at the time the review was left.
# (Uses the cutoff_2024, discussed above -- so the review deadline is slightly 
# later than the actual curriculum cutoff.)
mscs_reviews['required_when_taken'] = ((mscs_reviews['Timestamp'] < cutoff_2024) & mscs_reviews['2023_required']) | ((mscs_reviews['Timestamp'] >= cutoff_2024) & mscs_reviews['2024_required'])
mscs_reviews['outside'] = False

In [17]:
# Outside classes were never required, strictly speaking.
outside_reviews['required_when_taken'] = False
outside_reviews['outside'] = True

### Now the data can be combined into one dataset.

We can do some final data munging and drop unwanted features.

In [18]:
all_reviews = pd.concat([mscs_reviews, outside_reviews], join='outer')
all_reviews.to_csv(data_dir / 'all_reviews.csv', index=True)

In [19]:
keep_features = [
    # 'Timestamp',
    'overall_experience',
    'content_difficulty',
    'instructor_effectiveness',
    'time_commitment_hours',
    'required_when_taken',
    'final_weightage_pct',
    'final_exam',
    'final_exam_proctored',
    'final_assignment',
    'final_project',
    'outside'
]

all_reviews = all_reviews[all_reviews.columns.intersection(keep_features)]

In [20]:
all_reviews.corr()

,overall_experience,content_difficulty,instructor_effectiveness,time_commitment_hours,final_weightage_pct,final_exam,final_exam_proctored,final_assignment,final_project,required_when_taken,outside
overall_experience,1.000000,-0.051341,0.734055,0.058758,-0.204755,0.119854,-0.124525,-0.083253,-0.025600,-0.097315,0.092277
content_difficulty,-0.051341,1.000000,-0.106724,0.553381,-0.248488,-0.089316,0.048091,0.104568,-0.011469,0.242460,-0.190179
instructor_effectiveness,0.734055,-0.106724,1.000000,-0.058248,-0.291267,0.204188,-0.061104,-0.006163,-0.232534,-0.219129,0.184967
time_commitment_hours,0.058758,0.553381,-0.058248,1.000000,-0.085183,-0.289149,-0.012379,0.227276,0.066063,0.315777,-0.193267
final_weightage_pct,-0.204755,-0.248488,-0.291267,-0.085183,1.000000,-0.412533,-0.111364,0.041998,0.458995,-0.001987,0.133307
final_exam,0.119854,-0.089316,0.204188,-0.289149,-0.412533,1.000000,0.417660,-0.598889,-0.415286,-0.502375,0.247864
final_exam_proctored,-0.124525,0.048091,-0.061104,-0.012379,-0.111364,0.417660,1.000000,-0.269718,-0.184858,-0.445542,0.347901
final_assignment,-0.083253,0.104568,-0.006163,0.227276,0.041998,-0.598889,-0.269718,1.000000,-0.419871,0.557802,-0.375448
final_project,-0.025600,-0.011469,-0.232534,0.066063,0.458995,-0.415286,-0.184858,-0.419871,1.000000,-0.083910,0.150196
required_when_taken,-0.097315,0.242460,-0.219129,0.315777,-0.001987,-0.502375,-0.445542,0.557802,-0.083910,1.000000,-0.763196
